In [5]:
!pip install -U transformers datasets evaluate accelerate torch

  Using cached transformers-5.6.0-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.6.0-py3-none-any.whl (10.4 MB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl (530.6 MB)
Using cached huggingface_hub-1.11.0-py3-none-any.whl (645 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)


In [6]:
import os
import json
import time
import platform
import socket
import subprocess
from datetime import datetime

import requests
import torch
from transformers import AutoTokenizer, AutoModel

In [7]:

# ---------- Helpers ----------
def run(cmd):
    return subprocess.check_output(cmd, text=True).strip()

def get_imds_token(timeout=2):
    """Get IMDSv2 token (works when IMDS is enabled; required on many EC2 configs)."""
    try:
        r = requests.put(
            "http://169.254.169.254/latest/api/token",
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
            timeout=timeout,
        )
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    return None

def get_imds(path, token=None, timeout=2):
    """Query EC2 instance metadata. Returns None if not on EC2 / blocked."""
    url = f"http://169.254.169.254/latest/meta-data/{path}"
    headers = {}
    if token:
        headers["X-aws-ec2-metadata-token"] = token
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    return None

In [8]:
token = get_imds_token()
instance_id = get_imds("instance-id", token)
instance_type = get_imds("instance-type", token)
az = get_imds("placement/availability-zone", token)

print("=== EC2 Proof ===")
print("hostname:", socket.gethostname())
print("local time:", datetime.now().isoformat(timespec="seconds"))
print("instance-id:", instance_id)
print("instance-type:", instance_type)
print("availability-zone:", az)

if instance_id is None:
    print("WARNING: Could not access EC2 metadata. You might not be on EC2, or IMDS is disabled/blocked.")
else:
    print("OK: EC2 metadata accessible (strong proof you're on an EC2 instance).")

# Optional extra evidence: show OS + kernel
print("\n=== System ===")
print("platform:", platform.platform())
try:
    print("uname -a:", run(["uname", "-a"]))
except Exception:
    pass

=== EC2 Proof ===
hostname: 968c66602509
local time: 2026-04-22T22:26:31
instance-id: i-0a0b9d85dd9294ed7
instance-type: t2.medium
availability-zone: us-east-1c
OK: EC2 metadata accessible (strong proof you're on an EC2 instance).

=== System ===
platform: Linux-6.1.66-93.164.amzn2023.x86_64-x86_64-with-glibc2.35
uname -a: Linux 968c66602509 6.1.66-93.164.amzn2023.x86_64 #1 SMP PREEMPT_DYNAMIC Tue Jan  2 23:50:53 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux


In [9]:
print("\n=== Python & Packages ===")
print("python:", platform.python_version())
print("torch:", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("requests:", requests.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda version:", torch.version.cuda)
    print("gpu name:", torch.cuda.get_device_name(0))


=== Python & Packages ===
python: 3.11.6
torch: 2.11.0+cu130
transformers: 5.6.0
requests: 2.33.1
cuda available: False


In [11]:
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "hostname": socket.gethostname(),
    "instance_id": instance_id,
    "instance_type": instance_type,
    "availability_zone": az,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": __import__("transformers").__version__,
    "cuda_available": torch.cuda.is_available()
}

path = os.path.join(out_dir, "ec2_bert_smoketest.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("\nSaved proof file:", path)
print("Done ✅")


Saved proof file: outputs/ec2_bert_smoketest.json
Done ✅
